# Подробное описание решения

В этой задаче я делаю базовую аналитику по заказам пиццы из текстового журнала формата `дата;название;цена`.

**Цели анализа:**

1. Определить популярность каждой пиццы.
2. Посчитать выручку по датам.
3. Найти самый дорогой заказ.
4. Посчитать среднюю стоимость заказа.

**Как организовано решение:**

- Сначала парсю каждую строку и формирую единый список заказов в структурированном виде.
- Для популярности использую `Counter`.
- Для дневной выручки использую словарь с накоплением сумм.
- Для самого дорогого заказа применяю `max(..., key=...)`.
- Среднюю стоимость считаю как сумму цен, делённую на количество заказов.

**Почему такой вариант удобен:**

Код легко читается, а каждая метрика считается отдельно и прозрачно. Такой подход удобно расширять, если в следующих заданиях появятся новые KPI.


# Задача 5: Аналитика заказов пиццы

Нужно извлечь из записей о заказах:
1) список пицц и количество заказов (по убыванию популярности);
2) список дат и суммарную стоимость заказов в этот день (хронологически);
3) самый дорогой заказ;
4) среднюю стоимость заказа.

Формат входных данных зададим сами: каждая строка — один заказ
`дата;название;стоимость`

Дата в формате `YYYY-MM-DD`, стоимость — целое или вещественное число.


**Шаг 1. Считывание входных данных**

Для демонстрации используем набор строк с заказами.


In [1]:
# Пример входных данных (смешанные форматы даты и цены)
input_text = "2024-10-01;Маргарита;450\n01.10.2024;Пепперони;520,50\n2024/10/01;Маргарита;480.25\n02/10/2024;Гавайская;600\n2024-10-02;Пепперони;550\n2024-10-03;Маргарита;430,0\n"

lines = [line.strip() for line in input_text.strip().splitlines() if line.strip()]


**Шаг 2. Разбор записей**

Парсим строки в список заказов: (дата, название, стоимость).


In [2]:
from datetime import datetime
from decimal import Decimal, InvalidOperation

def normalize_date(raw_date):
    date_formats = (
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d.%m.%Y",
        "%d/%m/%Y",
        "%d-%m-%Y",
    )
    clean = raw_date.strip()
    for fmt in date_formats:
        try:
            return datetime.strptime(clean, fmt).strftime("%Y-%m-%d")
        except ValueError:
            pass
    return None

def parse_price(raw_price):
    clean = raw_price.strip().replace(' ', '').replace(',', '.')
    try:
        return Decimal(clean)
    except InvalidOperation:
        return None

orders = []
bad_lines = []

for line in lines:
    parts = [p.strip() for p in line.split(';')]
    if len(parts) != 3:
        bad_lines.append((line, 'неверное число полей'))
        continue

    raw_date, pizza_name, raw_price = parts
    date_str = normalize_date(raw_date)
    price = parse_price(raw_price)

    if date_str is None:
        bad_lines.append((line, 'некорректная дата'))
        continue
    if price is None:
        bad_lines.append((line, 'некорректная цена'))
        continue

    orders.append((date_str, pizza_name, price))


**Шаг 3. Список пицц по популярности**

Считаем количество заказов каждой пиццы и сортируем по убыванию.


In [3]:
from collections import Counter

pizza_counts = Counter(pizza for _, pizza, _ in orders)

popular_pizzas = sorted(
    pizza_counts.items(),
    key=lambda x: (-x[1], x[0])
)

print("Популярность пицц:")
for pizza, cnt in popular_pizzas:
    print(pizza, cnt)


Популярность пицц:
Маргарита 3
Пепперони 2
Гавайская 1


**Шаг 4. Список дат и суммарная выручка**

Суммируем стоимости по каждой дате и сортируем даты по возрастанию.


In [4]:
from collections import defaultdict

revenue_by_date = defaultdict(Decimal)
for date_str, _, price in orders:
    revenue_by_date[date_str] += price

sorted_dates = sorted(revenue_by_date.items())

print("Выручка по датам:")
for date_str, total in sorted_dates:
    print(date_str, f"{total:.2f}")


Выручка по датам:
2024-10-01 1450.75
2024-10-02 1150.00
2024-10-03 430.00


**Шаг 5. Самый дорогой заказ**

Находим запись с максимальной стоимостью.


In [5]:
if orders:
    max_order = max(orders, key=lambda x: x[2])
    print("Самый дорогой заказ:")
    print("Дата:", max_order[0])
    print("Пицца:", max_order[1])
    print("Стоимость:", f"{max_order[2]:.2f}")
else:
    print("Нет валидных заказов для анализа")


Самый дорогой заказ:
Дата: 2024-10-02
Пицца: Гавайская
Стоимость: 600.00


**Шаг 6. Средняя стоимость заказа**

Среднее арифметическое всех стоимостей.


In [6]:
if orders:
    avg_price = sum(price for _, _, price in orders) / Decimal(len(orders))
    print("Средняя стоимость заказа:", f"{avg_price:.2f}")
else:
    print("Средняя стоимость заказа: 0.00")

if bad_lines:
    print("Пропущено некорректных строк:", len(bad_lines))


Средняя стоимость заказа: 505.12
